### Why Do We Need Binarization?

**Binarization** is a technique where we take continuous numbers (or a wide range of numbers) and convert them into simple `0`s and `1`s based on a specific threshold. 

But why would we want to throw away detailed numbers and replace them with just 0 and 1?

#### 1. To Simplify the Problem (The "Titanic Family" Example)
In your notebook, you combined `SibSp` (siblings/spouses) and `Parch` (parents/children) into a single `family` column. The numbers in this column might range from `0` to `10`. 

However, sometimes the *exact number* doesn't actually matter to the model. What really matters is a simple Yes/No question: **"Was the passenger traveling alone, or did they have family with them?"**

By setting a binarization threshold at `0`:
* If `family == 0` → It stays `0` (Traveling Alone)
* If `family > 0` → It becomes `1` (Traveling with Family)

We have just turned a complex number into a simple "Yes/No" flag. This makes it incredibly easy for a machine learning model (like a Decision Tree) to understand the passenger's situation!

#### 2. To Reduce Noise
If you have a feature like "Number of times a user clicked a button", the difference between 1 click and 0 clicks is huge (they engaged vs. they didn't). But the difference between 99 clicks and 100 clicks is basically meaningless noise. Binarizing the data into "Clicked (1)" vs "Did Not Click (0)" removes that distracting noise.

#### 3. To Focus on Presence vs. Absence
Binarization is perfect whenever the **presence** of something is far more important than the **quantity**. 
* Do they have a family? (Yes/No)
* Did it rain today? (Yes/No - we might not care *how many inches* it rained, just that it happened).

**In short:** Binarization helps our machine learning models ignore distracting details and focus entirely on the core, fundamental differences in the data!


In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer

In [3]:
df = pd.read_csv('train.csv')[['Age','Fare','SibSp','Parch','Survived']]

In [4]:
df.dropna(inplace=True)

In [5]:
df.head()

,Age,Fare,SibSp,Parch,Survived
0,22.0,7.2500,1,0,0
1,38.0,71.2833,1,0,1
2,26.0,7.9250,0,0,1
3,35.0,53.1000,1,0,1
4,35.0,8.0500,0,0,0


In [6]:
df['family'] = df['SibSp'] + df['Parch']

In [7]:
df.head()

,Age,Fare,SibSp,Parch,Survived,family
0,22.0,7.2500,1,0,0,1
1,38.0,71.2833,1,0,1,1
2,26.0,7.9250,0,0,1,0
3,35.0,53.1000,1,0,1,1
4,35.0,8.0500,0,0,0,0


In [8]:
df.drop(columns=['SibSp','Parch'],inplace=True)

In [9]:
df.head()

,Age,Fare,Survived,family
0,22.0,7.2500,0,1
1,38.0,71.2833,1,1
2,26.0,7.9250,1,0
3,35.0,53.1000,1,1
4,35.0,8.0500,0,0


In [10]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [11]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [12]:
X_train.head()

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


In [13]:
# Without binarization

clf = DecisionTreeClassifier()

clf.fit(X_train,y_train)

y_pred = clf.predict(X_test)

accuracy_score(y_test,y_pred)

0.6433566433566433

In [14]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.6526995305164319)

In [ ]:
# Applying Binarization
from sklearn.preprocessing import Binarizer

In [16]:
trf = ColumnTransformer([
    ('bin',Binarizer(copy=False),['family'])
],remainder='passthrough')

In [17]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [18]:
pd.DataFrame(X_train_trf,columns=['family','Age','Fare'])

,family,Age,Fare
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [19]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_test,y_pred2)

0.6153846153846154

### What Does `cross_val_score` Do?

When you train a machine learning model, you usually split your data into a **Training Set** (to teach the model) and a **Testing Set** (to test it). But there is a hidden danger here: what if, purely by chance, your testing set happens to be full of super easy examples? Your model would score a 95% accuracy, making you think it's amazing. But if you fed it real-world data, it might crash and burn.

`cross_val_score` solves this problem by using a technique called **Cross-Validation**. 

Here is how it works step-by-step (using `cv=10`, which means 10 folds):

1. **Chop the Data:** It takes your *entire* dataset and chops it into 10 equal-sized chunks (called "folds").
2. **Train and Test (Round 1):** It holds out Chunk #1 to use as the Test Set. It uses Chunks #2 through #10 as the Training Set. It grades the model and records the score.
3. **Train and Test (Round 2):** It throws away the old model. Now, it holds out Chunk #2 as the Test Set, and trains on all the others. It grades the model and records the score.
4. **Repeat:** It repeats this process 10 times, so that *every single chunk* gets to be the Test Set exactly once.
5. **Final Grade:** It gives you a list of all 10 scores. Usually, we take the average (mean) of these 10 scores to get the final grade.

#### Why is this so important?

* **It Prevents "Lucky" Splits:** Because the model is tested on every single piece of data over the 10 rounds, it's impossible for it to get a high score just by getting a "lucky" train/test split.
* **It Measures Consistency:** If your 10 scores are wildly different (e.g., 90%, 40%, 85%, 30%), it tells you your model is highly unstable. If the scores are all around 80%, you know you have a reliable model.

**In short:** `cross_val_score` is the ultimate stress test for your machine learning model. It forces the model to take 10 different exams to prove how good it *really* is, rather than just taking one!


In [20]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X_trf,y,cv=10,scoring='accuracy'))

np.float64(0.6331964006259782)